# Commitment Bonds and Calibrated Trust

This computational benchmark compares cheap talk with the same promise backed by a transparent bond. It holds the hidden bot-type prior fixed at 50/50 and elicits a forecast before investment. Outputs are programmed examples, not human or LLM data.

**Decimal rule:** B promises $q(s)=3s/2$. Returns are exact half-points; nothing is rounded down.

In [ ]:
"""PS1 computational benchmark: transparent commitment bonds in a trust game.

This program creates simulated, programmed outcomes—not human or LLM evidence.
It keeps the bot-type distribution fixed while varying only whether a transparent
bond is attached to the same public promise.
"""
from fractions import Fraction
from pathlib import Path
import csv, json

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
PRIOR_HONEST = Fraction(1, 2)

def F(value):
    return value if isinstance(value, Fraction) else Fraction(str(value))

def promised_return(investment):
    """B promises half of B's tripled receipt: q(s) = 3s/2, with no rounding."""
    if type(investment) is not int or not 0 <= investment <= 10:
        raise ValueError("Investment must be an integer from 0 to 10")
    return Fraction(3 * investment, 2)

def payoffs(investment, returned, bond=0):
    """Return exact Fraction payoffs and the automatic penalty."""
    s, r, b = investment, F(returned), F(bond)
    if type(s) is not int or not 0 <= s <= 10:
        raise ValueError("Investment must be an integer from 0 to 10")
    if r < 0 or r > 3*s or r.denominator > 2:
        raise ValueError("Return must be from 0 to 3s in 0.5-point increments")
    if b < 0:
        raise ValueError("Bond coefficient cannot be negative")
    penalty = Fraction(0) if r == promised_return(s) else b*s
    return Fraction(10-s)+r+penalty, Fraction(3*s)-r-penalty, penalty

def bot_return(bot_type, condition, investment, bond=2):
    """Programmed benchmark behavior with type hidden from the participant."""
    if bot_type not in {"honest", "exploitative"}:
        raise ValueError("bot_type must be honest or exploitative")
    if condition not in {"cheap_talk", "bond"}:
        raise ValueError("condition must be cheap_talk or bond")
    if bot_type == "honest" or condition == "bond":
        return promised_return(investment)
    return Fraction(0)

def trial(bot_type, condition, forecast, investment, bond=2):
    """One proposed one-shot observation; forecast is elicited before investment."""
    p = F(forecast)
    if not 0 <= p <= 1:
        raise ValueError("Forecast must be between 0 and 1")
    r = bot_return(bot_type, condition, investment, bond)
    a, b_payoff, penalty = payoffs(investment, r, bond if condition == "bond" else 0)
    y = None if investment == 0 else int(r == promised_return(investment))
    return {"bot_type": bot_type, "condition": condition, "forecast": float(p),
            "investment": investment, "promised_return": float(promised_return(investment)),
            "return": float(r), "compliance": y, "payoff_A": float(a),
            "payoff_B": float(b_payoff), "welfare": float(a+b_payoff),
            "penalty": float(penalty), "brier": None if y is None else float((p-y)**2)}

def condition_benchmark(condition, investment=10, bond=2):
    """Expected compliance/payoffs under fixed 50/50 hidden bot types."""
    rows = [trial(t, condition, PRIOR_HONEST, investment, bond)
            for t in ("honest", "exploitative")]
    positive = [row for row in rows if row["compliance"] is not None]
    return {"condition": condition, "investment": investment, "bond": float(F(bond)),
            "type_prior_honest": float(PRIOR_HONEST),
            "expected_compliance": sum(row["compliance"] for row in positive)/len(positive),
            "expected_A_payoff": sum(row["payoff_A"] for row in rows)/len(rows),
            "expected_B_payoff": sum(row["payoff_B"] for row in rows)/len(rows),
            "expected_welfare": sum(row["welfare"] for row in rows)/len(rows)}

def calibration_summary(rows):
    """Planned-study calibration, using positive-investment trials only."""
    usable = [row for row in rows if row["compliance"] is not None]
    if not usable:
        return {"n": 0, "mean_forecast": None, "actual_compliance": None,
                "aggregate_calibration_gap": None, "mean_brier": None}
    mean_forecast = sum(row["forecast"] for row in usable)/len(usable)
    actual = sum(row["compliance"] for row in usable)/len(usable)
    return {"n": len(usable), "mean_forecast": mean_forecast, "actual_compliance": actual,
            "aggregate_calibration_gap": abs(mean_forecast-actual),
            "mean_brier": sum(row["brier"] for row in usable)/len(usable)}

# Verification of decimals and the incentive logic.
assert promised_return(7) == Fraction(21, 2)  # 10.5: half of B's 21-point receipt
assert F(7)/2 == Fraction(7, 2)               # 3.5: no floor operation
assert F(9)/2 == Fraction(9, 2)               # 4.5: no floor operation
assert bot_return("exploitative", "cheap_talk", 10) == 0
assert bot_return("exploitative", "bond", 10) == 15
assert payoffs(7, Fraction(21,2), 2) == (Fraction(27,2), Fraction(21,2), 0)
assert payoffs(7, 0, 2) == (17, 7, 14)

baseline = condition_benchmark("cheap_talk")
bonded = condition_benchmark("bond")
assert baseline["expected_compliance"] == 0.5
assert bonded["expected_compliance"] == 1.0

# Illustrative forecasts—synthetic, not human or LLM data.
demo_rows = [
    trial("honest", "cheap_talk", .80, 10), trial("exploitative", "cheap_talk", .80, 10),
    trial("honest", "bond", .95, 10), trial("exploitative", "bond", .95, 10),
]
summary = {"cheap_talk": calibration_summary(demo_rows[:2]),
           "bond": calibration_summary(demo_rows[2:])}

with (OUT/"condition_benchmarks.json").open("w") as f:
    json.dump({"baseline": baseline, "bond": bonded, "synthetic_calibration_demo": summary}, f, indent=2)
with (OUT/"synthetic_trials.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=demo_rows[0].keys())
    writer.writeheader(); writer.writerows(demo_rows)
print(json.dumps({"baseline": baseline, "bond": bonded, "synthetic_calibration_demo": summary}, indent=2))


## Eight-round educational extension

The one-shot benchmark above remains the project's clean causal comparison. This second component mirrors the interactive demo: five cheap-talk rounds followed by three bond rounds, using one fixed hidden bot type within a session. It is designed to visualize learning and pattern change, not to estimate a clean causal bond effect.

In [ ]:
def repeated_session(bot_type, forecasts, investments, bond=2):
    """Five cheap-talk rounds followed by three bond rounds for one fixed hidden type."""
    if len(forecasts) != 8 or len(investments) != 8:
        raise ValueError("Provide exactly eight forecasts and eight investments")
    rows = []
    for i, (forecast, investment) in enumerate(zip(forecasts, investments), start=1):
        condition = "cheap_talk" if i <= 5 else "bond"
        row = trial(bot_type, condition, forecast, investment, bond)
        row["round"] = i
        row["phase"] = "cheap talk" if i <= 5 else "bond"
        rows.append(row)
    return rows

def phase_summary(rows):
    result = {}
    for phase in ("cheap talk", "bond"):
        phase_rows = [r for r in rows if r["phase"] == phase]
        result[phase] = {
            **calibration_summary(phase_rows),
            "mean_investment": sum(r["investment"] for r in phase_rows) / len(phase_rows),
        }
    return result

# Synthetic example only: an exploitative bot breaks cheap-talk promises, then honors after the bond.
# Replace these lists with future participant or model forecasts/investments when collecting approved data.
session = repeated_session(
    bot_type="exploitative",
    forecasts=[.80, .70, .55, .40, .30, .45, .70, .88],
    investments=[6, 5, 4, 3, 2, 3, 5, 6],
)
session_summary = phase_summary(session)

with (OUT / "eight_round_synthetic_session.csv").open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=session[0].keys())
    writer.writeheader(); writer.writerows(session)
with (OUT / "eight_round_phase_summary.json").open("w") as f:
    json.dump(session_summary, f, indent=2)

for row in session:
    print(f"Round {row['round']} ({row['phase']}): forecast={row['forecast']:.0%}, investment={row['investment']}, compliance={row['compliance']}, Brier={row['brier']:.3f}")
print("\nPhase comparison (synthetic illustration):")
print(json.dumps(session_summary, indent=2))


## Interpretation

With investment fixed at 10, the programmed cheap-talk benchmark has 50% expected promise compliance, while the bond benchmark has 100%. Both have welfare of 30; the bond changes the payoff distribution and compliance rather than mechanically increasing welfare. The synthetic forecast rows illustrate how later participant data would be summarized with calibration gap and Brier score.